# COCO-pretrained vs. custom YOLO26m

The comparison uses **the same images, annotations and five categories**: `person`, `cat`, `dog`, `horse` and `cow`. In both ground truth and predictions, `civilian` and `rescuer` are merged into `person` before shared non-maximum suppression (NMS). Each prediction keeps the higher of the two class scores. This setup measures person detection across both roles.

The final comparison uses the test split and the shared five-class protocol. The earlier comparison of aggregate metrics across different class sets is archived separately.

## Prepare the evaluation split

The notebook defaults to the test split (183 images). The checkpoints and protocol were fixed using the validation split (383 images), which remains available for development. Keep the checkpoints and thresholds fixed when evaluating the test split.

Dataset preparation copies the selected split to a generated directory, validates the annotations, preserves bounding boxes and records hashes for the images and labels.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'dataset/data.yaml').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from evaluation.coco_comparison import prepare_dataset, run, plot_comparison, COMMON_NAMES

SPLIT = 'test'  # Use 'val' for development. The final protocol uses 'test'.
DEVICE = 'cpu'
BATCH = 8
common_yaml, dataset_manifest = prepare_dataset(SPLIT)
print('Classes:', COMMON_NAMES)
print('Images:', dataset_manifest['image_count'])
print('Objects:', dataset_manifest['annotation_count'])
print('Objects by class:', dataset_manifest['annotations_per_class'])
print('Dataset configuration:', common_yaml)

Classes: {0: 'person', 1: 'cat', 2: 'dog', 3: 'horse', 4: 'cow'}
Images: 183
Objects: 702
Objects by class: {'person': 451, 'dog': 52, 'cat': 40, 'horse': 78, 'cow': 81}
Dataset configuration: /home/carol/urban-disaster-monitor/notebooks/generated/coco_common_5class/test_full/data.yaml


## Shared evaluation protocol

Both checkpoints must use YOLO26m. The code checks that their scale and architecture match, with an allowed difference in the number of output classes.

- Use square 640 px inputs (`rect=False`) with the same batch size, device and FP32 precision.
- Use the prediction head with NMS enabled (`nms=True`) and test-time augmentation disabled.
- Exclude classes outside the five annotated categories before NMS for both models.
- Apply the same class-aware NMS rules: IoU 0.70, up to 300 detections and minimum confidence 0.001 for AP curves.
- Calculate micro-averaged precision and recall at **confidence 0.25 and matching IoU 0.50** by summing TP, FP and FN across all five classes.
- Use the same matching and AP implementation across IoU thresholds from 0.50 to 0.95.

The library reports macro-averaged precision and recall at the best F1 point. The result file stores those separately from the fixed-threshold micro averages. The AP curves start at confidence 0.001, while the fixed operating point uses 0.25.

The complete result is saved after both models finish. Quick verification runs use a separate file. The result records software versions, parameters, checkpoint hashes and the exact set of images and annotations.

In [ ]:
comparison = run(split=SPLIT, device=DEVICE, batch=BATCH)

import pandas as pd
from IPython.display import display
summary = []
for key, name in [('coco_pretrained', 'COCO-pretrained YOLO26m'), ('custom', 'Custom YOLO26m')]:
    row = comparison['models'][key]
    fixed = row['fixed_operating_point']
    summary.append({'Model': name,
                    'Micro precision (conf=0.25)': fixed['precision'],
                    'Micro recall (conf=0.25)': fixed['recall'],
                    'mAP@0.5': row['map50'], 'mAP@0.5:0.95': row['map50_95'],
                    'TP': fixed['tp'], 'FP': fixed['fp'], 'FN': fixed['fn']})
display(pd.DataFrame(summary).round(4))

## Review per-class metrics

Use the per-class AP scores alongside the aggregate metrics. The `person` score covers civilians and rescuers together.

In [ ]:
per_class = []
for class_name in COMMON_NAMES.values():
    item = {'Class': class_name}
    for key, name in [('coco_pretrained', 'COCO'), ('custom', 'Custom')]:
        row = comparison['models'][key]
        item[f'{name} AP@0.5'] = row['ap50_per_class'].get(class_name)
        item[f'{name} AP@0.5:0.95'] = row['ap50_95_per_class'].get(class_name)
    per_class.append(item)
display(pd.DataFrame(per_class).round(4))

## Generate the chart from saved results

Run this cell on its own to export the complete five-class comparison as a 300 dpi PNG and a PDF in `notebooks/generated/figures/`.

The chart shows **micro-averaged precision and recall at confidence 0.25 and matching IoU 0.50**. The mAP values use the AP curves for all five classes.

In [ ]:
from pathlib import Path
import json, sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'dataset/data.yaml').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from evaluation.coco_comparison import plot_comparison
import matplotlib.pyplot as plt
SAVED_SPLIT = 'test'
cache = PROJECT_ROOT / 'models/comparisons' / f'coco-vs-custom-yolo26m-common-{SAVED_SPLIT}.json'
if not cache.is_file():
    raise FileNotFoundError('Run the complete five-class evaluation before generating the chart.')
saved = json.loads(cache.read_text())
fig = plot_comparison(saved)
plt.show()

## Interpret the results

Both checkpoints use the same class definitions, input geometry, postprocessing and evaluator. The custom checkpoint was adapted to the disaster dataset and selected using its validation split. The comparison measures these two checkpoints. It cannot isolate the effect of the dataset, initialization, augmentation or training duration.

The checkpoints, thresholds and protocol were fixed before test evaluation. Follow-up analysis could estimate confidence intervals by resampling images, review errors by class and compare real and synthetic images when per-image provenance is available. Overlap between publicly sourced images and pretraining data was not audited.

The result file records the Ultralytics version. The adapter uses its validation API and checks tensor shapes. Verify the procedure again after upgrading Ultralytics.